In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
path = '/content/drive/MyDrive/Colab Notebooks/'

In [4]:
import pandas as pd
import numpy as np

In [5]:
movies = pd.read_csv(path + 'movies.dat', sep='::', header=None,
                     names=['movieId', 'title', 'genres'],
                     engine='python', encoding='ISO-8859-1')

ratings = pd.read_csv(path + 'ratings.dat', sep='::', header=None,
                      names=['userId', 'movieId', 'rating', 'timestamp'],
                      engine='python', encoding='ISO-8859-1')

users = pd.read_csv(path + 'users.dat', sep='::', header=None,
                    names=['userId', 'gender', 'age', 'occupation', 'zip'],
                    engine='python', encoding='ISO-8859-1')

In [6]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [8]:
users.head()

,userId,gender,age,occupation,zip
0,1,F,1,10,48067
1,2,M,56,16,70072
2,3,M,25,15,55117
3,4,M,45,7,02460
4,5,M,25,20,55455


In [9]:
user_movie_matrix = ratings.pivot_table(index='userId', columns='movieId', values='rating')

In [10]:
movie_user_matrix = user_movie_matrix.T

In [11]:
movie_counts = ratings['movieId'].value_counts()
popular_movies = movie_counts[movie_counts > 500].index

filtered_matrix = movie_user_matrix.loc[popular_movies]
filtered_matrix.shape

(617, 6040)

In [12]:
movie_similarity = filtered_matrix.T.corr()

In [13]:
movie_similarity.head()

movieId,2858,260,1196,1210,480,2028,589,2571,1270,593,...,1956,1635,1982,2469,3534,2917,3701,3654,58,838
movieId,,,,,,,,,,,,,,,,,,,,,
2858,1.000000,0.068348,0.089290,0.103226,-0.003588,0.154980,0.055629,0.142432,0.032069,0.155786,...,0.133217,0.341255,0.087454,0.200122,0.082374,0.249167,0.144194,0.079094,0.172039,0.074324
260,0.068348,1.000000,0.661552,0.574808,0.240746,0.146365,0.191322,0.234341,0.259374,0.121831,...,0.085212,-0.035616,0.072799,0.080060,-0.016919,0.119250,0.244883,0.168568,-0.005844,0.019524
1196,0.089290,0.661552,1.000000,0.631437,0.201458,0.120312,0.218605,0.208029,0.273120,0.114281,...,0.101887,-0.014429,0.124069,0.045523,0.080539,0.042676,0.155450,0.140911,0.012218,0.068575
1210,0.103226,0.574808,0.631437,1.000000,0.307364,0.169816,0.256786,0.217053,0.288792,0.122195,...,0.092230,-0.087959,0.051800,0.071715,0.055129,0.075833,0.189235,0.209709,0.099186,-0.057370
480,-0.003588,0.240746,0.201458,0.307364,1.000000,0.228763,0.308324,0.163542,0.313988,0.206499,...,0.091265,-0.068930,0.240919,0.179085,0.176526,0.073431,0.267494,0.147511,0.015134,0.038613


In [14]:
movies[movies['title'].str.contains('Toy Story', case=False, na=False)]

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
3045,3114,Toy Story 2 (1999),Animation|Children's|Comedy


In [15]:
similar_movies = movie_similarity[1].sort_values(ascending=False)
similar_movies.head(10)

,1
movieId,
1,1.000000
3114,0.630386
2355,0.449198
588,0.429626
595,0.419857
364,0.388768
1022,0.387174
2018,0.376215
3396,0.369807


In [16]:
similar_movies = similar_movies.drop(1)
similar_movies.head(10)

,1
movieId,
3114,0.630386
2355,0.449198
588,0.429626
595,0.419857
364,0.388768
1022,0.387174
2018,0.376215
3396,0.369807
2081,0.368345


In [17]:
recommendations = similar_movies.head(10).reset_index()
recommendations.columns = ['movieId', 'similarity']

recommendations = recommendations.merge(movies, on='movieId')
recommendations[['title', 'genres', 'similarity']]

,title,genres,similarity
0,Toy Story 2 (1999),Animation|Children's|Comedy,0.630386
1,"Bug's Life, A (1998)",Animation|Children's|Comedy,0.449198
2,Aladdin (1992),Animation|Children's|Comedy|Musical,0.429626
3,Beauty and the Beast (1991),Animation|Children's|Musical,0.419857
4,"Lion King, The (1994)",Animation|Children's|Musical,0.388768
5,Cinderella (1950),Animation|Children's|Musical,0.387174
6,Bambi (1942),Animation|Children's,0.376215
7,"Muppet Movie, The (1979)",Children's|Comedy,0.369807
8,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance,0.368345
9,101 Dalmatians (1961),Animation|Children's,0.364210


In [18]:
def recommend(movie_name):
    movie = movies[movies['title'].str.contains(movie_name, case=False)]
    if movie.empty:
        return "Movie not found"

    movie_id = movie.iloc[0]['movieId']

    if movie_id not in movie_similarity.columns:
        return "Movie not in filtered dataset"

    similar = movie_similarity[movie_id].sort_values(ascending=False)[1:11]

    result = similar.reset_index()
    result.columns = ['movieId', 'similarity']

    result = result.merge(movies, on='movieId')

    return result[['title', 'genres', 'similarity']]

In [19]:
recommend("Toy Story")

,title,genres,similarity
0,Toy Story 2 (1999),Animation|Children's|Comedy,0.630386
1,"Bug's Life, A (1998)",Animation|Children's|Comedy,0.449198
2,Aladdin (1992),Animation|Children's|Comedy|Musical,0.429626
3,Beauty and the Beast (1991),Animation|Children's|Musical,0.419857
4,"Lion King, The (1994)",Animation|Children's|Musical,0.388768
5,Cinderella (1950),Animation|Children's|Musical,0.387174
6,Bambi (1942),Animation|Children's,0.376215
7,"Muppet Movie, The (1979)",Children's|Comedy,0.369807
8,"Little Mermaid, The (1989)",Animation|Children's|Comedy|Musical|Romance,0.368345
9,101 Dalmatians (1961),Animation|Children's,0.364210


In [21]:
movies.to_pickle("movies.pkl")
movie_similarity.to_pickle("movie_similarity.pkl")

In [22]:
from google.colab import files

files.download("movies.pkl")
files.download("movie_similarity.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>